# 🏠 House Price Prediction
### End-to-End ML Project | Python 3.13 Compatible
---
**Tech Stack:** Python · Pandas · NumPy · Scikit-learn · Matplotlib · Seaborn

**Models:** Linear Regression · Decision Tree · Random Forest

**Target:** Predict property price in Lakhs (₹)

## 1. Setup & Imports

In [ ]:
import sys
print(f'Python version: {sys.version}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
print('All imports successful ✓')

## 2. Generate Synthetic Dataset

In [ ]:
np.random.seed(42)
n = 1000

area_sqft      = np.random.randint(500, 5000, n)
bedrooms       = np.random.choice([1,2,3,4,5], n, p=[0.1,0.3,0.35,0.15,0.1])
bathrooms      = np.clip(bedrooms - 1 + np.random.randint(0,2,n), 1, 5)
age_years      = np.random.randint(0, 30, n)
parking        = np.random.choice([0,1,2], n, p=[0.2,0.6,0.2])
location_score = np.random.randint(1, 11, n)
furnishing     = np.random.choice(['unfurnished','semi-furnished','furnished'], n, p=[0.3,0.4,0.3])
property_type  = np.random.choice(['apartment','villa','independent house'], n, p=[0.55,0.2,0.25])

furn_val = {'unfurnished':0,'semi-furnished':1,'furnished':2}
prop_val = {'apartment':0,'independent house':1,'villa':2}

f = np.array([furn_val[x] for x in furnishing])
p = np.array([prop_val[x] for x in property_type])

price = (area_sqft*0.03 + bedrooms*5 + bathrooms*3 +
         location_score*8 + parking*4 + f*6 + p*12 -
         age_years*0.5 + np.random.normal(0,8,n))

price_lakhs = np.clip(price, 10, 500).round(2)

df = pd.DataFrame({
    'area_sqft': area_sqft, 'bedrooms': bedrooms,
    'bathrooms': bathrooms, 'age_years': age_years,
    'parking': parking, 'location_score': location_score,
    'furnishing': furnishing, 'property_type': property_type,
    'price_lakhs': price_lakhs
})

print('Dataset shape:', df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print(df.describe().round(2))
print('\nNull values:\n', df.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['price_lakhs'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price (Lakhs)')
axes[1].hist(df['area_sqft'], bins=40, color='coral', edgecolor='white')
axes[1].set_title('Area Distribution')
axes[1].set_xlabel('Area (sq ft)')
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10,7))
sns.heatmap(df.select_dtypes(include='number').corr(),
            annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
sns.boxplot(data=df, x='furnishing', y='price_lakhs', ax=axes[0],
            order=['unfurnished','semi-furnished','furnished'], palette='pastel')
axes[0].set_title('Price by Furnishing')
sns.boxplot(data=df, x='property_type', y='price_lakhs', ax=axes[1], palette='pastel')
axes[1].set_title('Price by Property Type')
plt.tight_layout(); plt.show()

## 4. Preprocessing & Feature Engineering

In [ ]:
df2 = df.copy()

le_f = LabelEncoder()
le_p = LabelEncoder()
df2['furnishing_enc']    = le_f.fit_transform(df2['furnishing'])
df2['property_type_enc'] = le_p.fit_transform(df2['property_type'])

df2['room_ratio']  = (df2['bathrooms'] / df2['bedrooms']).round(3)
df2['value_index'] = (df2['location_score'] * df2['area_sqft'] / 1000).round(3)

feature_cols = ['area_sqft','bedrooms','bathrooms','age_years',
                'parking','location_score','furnishing_enc',
                'property_type_enc','room_ratio','value_index']

X = df2[feature_cols]
y = df2['price_lakhs']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 5. Model Training & Evaluation

In [ ]:
def evaluate(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    print(f'{name:25s}  MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}')
    return model, preds

lr_model,  lr_preds  = evaluate('Linear Regression',  LinearRegression(),              X_train, y_train, X_test, y_test)
dt_model,  dt_preds  = evaluate('Decision Tree',       DecisionTreeRegressor(max_depth=8, random_state=42), X_train, y_train, X_test, y_test)
rf_model,  rf_preds  = evaluate('Random Forest',       RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), X_train, y_train, X_test, y_test)

## 6. Visualization — Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(y_test, rf_preds, alpha=0.5, s=20, color='steelblue')
mn, mx = y_test.min(), y_test.max()
plt.plot([mn,mx],[mn,mx],'r--', label='Perfect Prediction')
plt.xlabel('Actual Price (Lakhs)')
plt.ylabel('Predicted Price (Lakhs)')
plt.title('Actual vs Predicted — Random Forest')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
fi = pd.DataFrame({'Feature': feature_cols, 'Importance': rf_model.feature_importances_})
fi = fi.sort_values('Importance', ascending=True)
plt.figure(figsize=(8,5))
plt.barh(fi['Feature'], fi['Importance'], color='steelblue')
plt.title('Feature Importance — Random Forest')
plt.tight_layout(); plt.show()

## 7. Predict New Property Price

In [ ]:
# Enter your property details here
my_property = {
    'area_sqft':      1800,
    'bedrooms':       3,
    'bathrooms':      2,
    'age_years':      5,
    'parking':        1,
    'location_score': 7,
    'furnishing_enc': 1,     # 0=unfurnished, 1=semi, 2=furnished
    'property_type_enc': 0,  # 0=apartment, 1=independent, 2=villa
    'room_ratio':     2/3,
    'value_index':    7*1800/1000
}

X_new = scaler.transform([list(my_property.values())])
price = rf_model.predict(X_new)[0]
print(f'Predicted Price: ₹{price:.2f} Lakhs')